# Tutorial 04 — Active and Passive Stress

The notebook computes every curve from the local `active_mechanics.py` module; committed figures are not loaded.

In [ ]:
LANGUAGE = "en"

import importlib.util
import os
import sys
from pathlib import Path


def _is_repository_root(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (
        path / "src" / "biomechanics_tutorials"
    ).is_dir()


def _find_repository_root() -> Path:
    candidates = []
    installed_spec = importlib.util.find_spec("biomechanics_tutorials")
    if installed_spec is not None and installed_spec.origin:
        package_file = Path(installed_spec.origin).resolve()
        if len(package_file.parents) >= 3:
            candidates.append(package_file.parents[2])
    environment_root = os.environ.get("BMRT_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    home = Path.home()
    for directory in [home, home / "Desktop", home / "Documents", home / "Downloads"]:
        candidates.append(directory / "Biomechanics-Research-Tutorials")
        if directory.is_dir():
            candidates.extend(directory.glob("Biomechanics-Research-Tutorials*"))
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_repository_root(candidate):
            return candidate
    raise RuntimeError(
        "Repository root was not found. Extract the complete repository or install it with python -m pip install -e .[dev]"
    )


REPOSITORY_ROOT = _find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import numpy as np
import matplotlib.pyplot as plt
import biomechanics_tutorials
from biomechanics_tutorials.active_mechanics import (
    activation_biexponential,
    activation_half_cosine,
    calcium_gamma_transient,
    force_length,
    force_velocity,
    active_tension,
    passive_uniaxial_nominal_stress,
    active_stress_uniaxial_nominal_response,
    active_strain_uniaxial_nominal_response,
    calibrate_active_stretch,
    active_stress_shear_response,
    active_strain_shear_response,
    simulate_crossbridge,
    simulate_isometric_twitch,
    simulate_isotonic_shortening,
)
from biomechanics_tutorials.plotting import apply_tutorial_style

apply_tutorial_style()
print("Repository root:", REPOSITORY_ROOT)
print("Python kernel:", sys.executable)
print("Local package:", Path(biomechanics_tutorials.__file__).resolve())

## Activation

In [ ]:
time = np.linspace(0.0, 0.8, 800)
activation = activation_biexponential(time)
plt.plot(time, activation, label="Bi-exponential")
plt.plot(time, activation_half_cosine(time), label="Half-cosine")
plt.plot(time, calcium_gamma_transient(time), label="Calcium-like")
plt.xlabel("Time" if LANGUAGE == "en" else "Время")
plt.ylabel("Activation" if LANGUAGE == "en" else "Активация")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Force–length and force–velocity

In [ ]:
stretch = np.linspace(0.7, 1.5, 400)
velocity = np.linspace(-1.0, 0.95, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(stretch, force_length(stretch))
axes[0].set_title("Force-length" if LANGUAGE == "en" else "Сила–длина")
axes[1].plot(velocity, force_velocity(velocity))
axes[1].set_title("Force-velocity" if LANGUAGE == "en" else "Сила–скорость")
for axis in axes:
    axis.grid(alpha=0.25)
plt.show()

## Stress decomposition

In [ ]:
passive = passive_uniaxial_nominal_stress(stretch)
tension = active_tension(0.75, stretch)
active_nominal = tension / stretch
plt.plot(stretch, passive, label="Passive")
plt.plot(stretch, active_nominal, label="Active")
plt.plot(stretch, passive + active_nominal, label="Total")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Calibrating active stress and active strain

In [ ]:
reference_stretch = 1.12
tension_level = 26.0
active_stretch = calibrate_active_stretch(reference_stretch, tension_level / reference_stretch)
stress_curve = active_stress_uniaxial_nominal_response(stretch, tension_level)
strain_curve = active_strain_uniaxial_nominal_response(stretch, active_stretch)
print("Calibrated active stretch:", active_stretch)
plt.plot(stretch, stress_curve, label="Active stress")
plt.plot(stretch, strain_curve, label="Active strain")
plt.axvline(reference_stretch, linestyle="--", alpha=0.6)
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Shear comparison

In [ ]:
gamma = np.linspace(-0.6, 0.6, 300)
plt.plot(gamma, active_stress_shear_response(gamma, tension_level), label="Active stress")
plt.plot(gamma, active_strain_shear_response(gamma, active_stretch), label="Active strain")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Calcium and cross-bridges

In [ ]:
calcium_target = calcium_gamma_transient(time)
kinetics = simulate_crossbridge(time, calcium_target)
plt.plot(time, kinetics["calcium"], label="Calcium")
plt.plot(time, kinetics["bound_fraction"], label="Bound cross-bridges")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Isometric twitch

In [ ]:
isometric = simulate_isometric_twitch(time, 1.12, activation)
plt.plot(time, isometric["passive"], label="Passive")
plt.plot(time, isometric["active"], label="Active")
plt.plot(time, isometric["total"], label="Total")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Isotonic shortening

In [ ]:
for afterload in [10.0, 22.0, 34.0]:
    result = simulate_isotonic_shortening(time, activation, afterload)
    plt.plot(time, result["stretch"], label=f"Afterload {afterload:g}")
plt.legend()
plt.grid(alpha=0.25)
plt.show()